# Proyecto 2 — Detección de Neumonía en Radiografías Torácicas
## Notebook 1: Análisis Exploratorio de Datos (EDA)

**Objetivo:** Comprender la distribución, calidad y características del dataset de imágenes de rayos X para diseñar la estrategia de preprocesamiento y augmentación.

**Dataset:** Chest X-Ray Images (Pneumonia) — Kaggle / NIH  
**Clases:** NORMAL vs PNEUMONIA  
**Tamaño:** 5.863 imágenes

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

from utils import CLASSES, load_image

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR = Path('../data/raw')
print('Setup completo.')

## 1. Inventario del dataset

In [ ]:
splits = ['train', 'val', 'test']
inventory = {}
for split in splits:
    inventory[split] = {}
    for cls in CLASSES:
        class_dir = DATA_DIR / split / cls
        if class_dir.exists():
            n = len(list(class_dir.glob('*.jpeg')) + list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')))
        else:
            n = 0
        inventory[split][cls] = n

inv_df = pd.DataFrame(inventory).T
inv_df['Total'] = inv_df.sum(axis=1)
inv_df['% PNEUMONIA'] = (inv_df['PNEUMONIA'] / inv_df['Total'] * 100).round(1)
print(inv_df)

# Visualizar distribución
fig, axes = plt.subplots(1, len(splits), figsize=(15, 5))
colors = ['steelblue', 'coral']
for ax, split in zip(axes, splits):
    values = list(inventory[split].values())
    wedges, texts, autotexts = ax.pie(
        values, labels=CLASSES, autopct='%1.1f%%',
        colors=colors, startangle=90
    )
    ax.set_title(f'{split.capitalize()} (n={sum(values)})', fontsize=12, fontweight='bold')

plt.suptitle('Distribución de clases por split', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Visualización de muestras representativas

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))

for row, cls in enumerate(CLASSES):
    cls_dir = DATA_DIR / 'train' / cls
    images = sorted(cls_dir.glob('*.jpeg'))[:5] if cls_dir.exists() else []
    for col, img_path in enumerate(images[:5]):
        img = Image.open(img_path).convert('RGB').resize((224, 224))
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(f'{cls}\n{img_path.name[:20]}', fontsize=8)
        axes[row, col].axis('off')

plt.suptitle('Muestras representativas — NORMAL (arriba) vs PNEUMONIA (abajo)', fontsize=13)
plt.tight_layout()
plt.show()

**Observaciones visuales:**
- Las imágenes **NORMAL** muestran pulmones claros, sin opacidades
- Las imágenes de **PNEUMONIA** muestran opacidades, consolidaciones o infiltrados
- Hay variabilidad en orientación, escala y contraste → justifica data augmentation

## 3. Análisis de resolución y tamaño de imágenes

In [ ]:
resolutions = []
for cls in CLASSES:
    cls_dir = DATA_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    for p in list(cls_dir.glob('*.jpeg'))[:200]:  # Sample 200 per class
        try:
            img = Image.open(p)
            w, h = img.size
            resolutions.append({'class': cls, 'width': w, 'height': h,
                                 'file_size_kb': p.stat().st_size / 1024})
        except Exception:
            pass

res_df = pd.DataFrame(resolutions)
if not res_df.empty:
    print(res_df.groupby('class')[['width', 'height', 'file_size_kb']].describe().round(1))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for cls, grp in res_df.groupby('class'):
        axes[0].scatter(grp['width'], grp['height'], alpha=0.3, label=cls, s=20)
    axes[0].set_xlabel('Ancho (px)')
    axes[0].set_ylabel('Alto (px)')
    axes[0].set_title('Distribución de resoluciones')
    axes[0].legend()

    res_df.groupby('class')['file_size_kb'].plot(kind='hist', alpha=0.6, bins=30, ax=axes[1])
    axes[1].set_xlabel('Tamaño de archivo (KB)')
    axes[1].set_title('Distribución de tamaños de archivo')
    axes[1].legend(CLASSES)

    plt.tight_layout()
    plt.show()

## 4. Análisis de histogramas de píxeles

In [ ]:
pixel_data = {cls: [] for cls in CLASSES}
for cls in CLASSES:
    cls_dir = DATA_DIR / 'train' / cls
    if not cls_dir.exists():
        continue
    for p in list(cls_dir.glob('*.jpeg'))[:50]:
        try:
            arr = np.array(Image.open(p).convert('L').resize((64, 64)), dtype=np.float32) / 255.0
            pixel_data[cls].append(arr.mean())  # Mean intensity per image
        except Exception:
            pass

fig, ax = plt.subplots(figsize=(10, 5))
for cls, means in pixel_data.items():
    if means:
        ax.hist(means, bins=25, alpha=0.6, label=cls, edgecolor='white')
ax.set_xlabel('Intensidad media de píxeles (normalizada)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de intensidad media por clase', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Plan de preprocesamiento y augmentación

Basado en el análisis exploratorio, definimos la estrategia:

| Técnica | Justificación |
|---------|---------------|
| Resize a 224×224 | Compatibilidad con ResNet50 preentrenada en ImageNet |
| Normalización /255.0 | Escalar a [0,1] para convergencia estable |
| Flip horizontal | Las radiografías son simétricas — augmentación válida |
| Rotación ±15° | Variabilidad en posicionamiento del paciente |
| Zoom ±10% | Diferentes distancias de la fuente de rayos X |
| `class_weight` | Dataset desbalanceado (27% NORMAL / 73% PNEUMONIA) |

**Próximo notebook:** `02_cnn_transfer_learning.ipynb`